In [1]:
# IMPORTS
import os
import requests
import pandas as pd
import time
from dotenv import load_dotenv
from sqlalchemy import create_engine,text


# LOAD VARIABLES
load_dotenv()


# CONFIG
INITIAL_LEGISLATURE = 54
FINAL_LEGISLATURE = 57
TABLE_NAME = "deputy_history"
BASE_URL = ("https://dadosabertos.camara.leg.br/api/v2/deputados")


In [2]:
# GET LEGISLATURES
def get_deputy_history(deputy_id):

    print(f"Collecting deputy history for deputy {deputy_id}")

    url = f"{BASE_URL}/{deputy_id}/historico"

    max_retries = 10

    for attempt in range(max_retries):

        try:

            response = requests.get(
                url,
                timeout=600
            )

            if response.status_code == 504:
                raise Exception("Gateway Timeout")

            response.raise_for_status()

            data = response.json()

            page_data = data.get("dados", [])

            if not page_data:
                return pd.DataFrame()

            return pd.DataFrame(page_data)

        except Exception as e:

            print(
                f"Attempt {attempt + 1} failed "
                f"for deputy {deputy_id}: {e}"
            )

            time.sleep(2)

    return pd.DataFrame()


# SAVE PARQUET FUNCTION
def save_parquet(df, table_name):

    # Create directory
    path = f"data/raw/{table_name}"

    os.makedirs(path, exist_ok=True)

    # File path
    file_path = (
        f"{path}/{table_name}.parquet"
    )

    # Save parquet
    df.to_parquet(file_path,index=False)

    print(f"Parquet saved at: {file_path}")


def get_deputy_ids(engine):

    query = text("""
        SELECT DISTINCT id
        FROM bronze.legislatures
        WHERE 1=1
        LIMIT 10000000
    """)

    with engine.connect() as conn:
        result = conn.execute(query)
        return [row[0] for row in result]

In [3]:
engine = create_engine(os.getenv("DATABASE_CONNECTION"))

deputy_list = get_deputy_ids(engine)

display(deputy_list)



[74152,
 179384,
 168034,
 205234,
 121948,
 220621,
 73424,
 141517,
 193726,
 220704,
 204511,
 66828,
 204457,
 204538,
 178848,
 73874,
 160638,
 73584,
 204385,
 220654,
 122974,
 160596,
 178959,
 220547,
 224333,
 200153,
 74430,
 74352,
 164361,
 171622,
 204504,
 141375,
 230767,
 178885,
 74345,
 219592,
 178860,
 178843,
 160568,
 178864,
 220530,
 141519,
 74383,
 214865,
 74160,
 141558,
 204498,
 161905,
 204479,
 217036,
 141450,
 75431,
 103758,
 178856,
 160529,
 215044,
 178802,
 233598,
 73831,
 141532,
 220575,
 178862,
 204517,
 178987,
 220568,
 178950,
 220648,
 227324,
 227660,
 229333,
 160679,
 74095,
 220677,
 220638,
 74048,
 73434,
 204509,
 165583,
 219585,
 133372,
 160602,
 178851,
 220553,
 160595,
 160645,
 160668,
 185500,
 178846,
 74238,
 171786,
 204487,
 73772,
 160976,
 220008,
 141467,
 74833,
 141508,
 137983,
 74686,
 204426,
 132504,
 141510,
 141484,
 73682,
 160664,
 220574,
 205303,
 160615,
 204481,
 166402,
 131943,
 74382,
 204384,
 179

In [ ]:


engine = create_engine(os.getenv("DATABASE_CONNECTION"))

deputy_list = get_deputy_ids(engine)

rows = []

for deputy_id in deputy_list:

    data = get_deputy_history(deputy_id)

    if not data.empty:
        rows.append(data)

if not rows:
    print("No data collected.")
    
# Concatenating dataframes
final_df = pd.concat(rows, ignore_index=True)

print(f"Total rows collected: {len(final_df)}")

save_parquet(final_df, TABLE_NAME)

Attempt 1 failed for deputy 129618: Gateway Timeout
Attempt 2 failed for deputy 129618: Gateway Timeout
Attempt 3 failed for deputy 129618: Gateway Timeout
